# HW3 Lab: Text Processing with spaCy and NLTK 🧹📝

<!-- lab-header:begin -->
**MIS 769 · Big Data Analytics for Business · Fall 2026**

**Points:** 40  |  **Due:** Sunday, September 20, 2026 at 10:00 PM Pacific

**Dr. Richard J. Young** · UNLV Lee Business School
<!-- lab-header:end -->

> **Why are we learning this?** 80% of business data is text — reviews, tickets, emails,
> contracts — and every text pipeline starts with cleaning decisions that quietly decide
> what your analysis can see. The transferable skill is **making preprocessing choices
> deliberately**: knowing what each cleaning step removes, and when removing it destroys
> the very signal you're after (spoiler: "not" matters in reviews).

**How this lab works:** everything below is a *working demo* — hit Runtime → Run all and
the whole pipeline succeeds on Yelp reviews. Your job is to (1) understand each step,
(2) twist the knobs in the 🧪 Experiment cells, and (3) re-run the pipeline on **text
from a domain you're passionate about** in the 🎮 Playground. That second run is where
your grade lives.

**The pipeline:** Load text → Standard cleaning → Domain-specific cleaning → Know when cleaning HURTS → Visualize the difference


## Learning objectives

- Set up spaCy and NLTK in Colab and apply standard stopword removal
- Build a **domain-specific** stopword list and justify every word in it
- Diagnose the **negation problem** and build a "smart" list that preserves not/never
- Show the impact of each cleaning choice with a side-by-side word cloud


## Part 1-2: Setup and Data (6 pts)

One pip line, then downloads. Two gotchas current guides get wrong: NLTK ≥3.8.2 renamed
its tokenizer data to `punkt_tab` (downloading `punkt` no longer works), and the spaCy
English model is installed with a shell command, not pip's package name.


In [ ]:
!pip install -q spacy nltk wordcloud datasets pandas matplotlib
!python -m spacy download en_core_web_sm -q

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from collections import Counter
import re

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)   # NLTK >=3.8.2 renamed 'punkt' data
from nltk.corpus import stopwords

import spacy
nlp = spacy.load("en_core_web_sm")

STANDARD_STOPWORDS = set(stopwords.words("english"))
print(f"✅ spaCy {spacy.__version__} | nltk {nltk.__version__}")
print(f"✅ {len(STANDARD_STOPWORDS)} standard English stopwords")
print(f"   Examples: {sorted(STANDARD_STOPWORDS)[:10]}")

### Load the data

Default: **Yelp reviews** (real customer language, star ratings included — perfect for
seeing what cleaning does to sentiment). Bounded to 20,000 rows so everything runs in
minutes on free Colab. The 🎮 Playground at the end swaps this for text from YOUR world —
just set `text_column` / `label_column` to match whatever you load.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("Yelp/yelp_review_full", split="train[:20000]")
df = dataset.to_pandas()
text_column, label_column = "text", "label"   # label = 0-4 stars (0 = 1 star)

# Alternatives:
#   dataset = load_dataset("stanfordnlp/imdb", split="train[:20000]")
#   or upload your own CSV and set the column names above

print(f"✅ Loaded {len(df):,} reviews")
print("Columns:", list(df.columns))
print("\nSample:", df[text_column].iloc[0][:200], "...")

## Part 3: Standard Stopwords (8 pts)

Stopword removal strips "the, is, at, and…" — words so common they (usually) carry no
signal. First we apply the standard list and **quantify** what it removes. Never clean
blind: always measure what a cleaning step actually did to your data.


In [ ]:
def remove_stopwords(text, stopwords_set):
    """Lowercase, split on whitespace, drop stopwords."""
    words = str(text).lower().split()
    return " ".join(w for w in words if w not in stopwords_set)

df["text_original"] = df[text_column].astype(str)
df["text_cleaned"] = df["text_original"].apply(lambda x: remove_stopwords(x, STANDARD_STOPWORDS))

df["word_count_original"] = df["text_original"].str.split().str.len()
df["word_count_cleaned"] = df["text_cleaned"].str.split().str.len()
pct = 100 - df["word_count_cleaned"].sum() / df["word_count_original"].sum() * 100

print("📊 STOPWORD REMOVAL IMPACT")
print(f"   Before: {df['word_count_original'].sum():,} words")
print(f"   After:  {df['word_count_cleaned'].sum():,} words  ({pct:.1f}% removed)")

# Eyeball one before/after — always look at actual examples, not just totals
i = 0
print(f"\nORIGINAL: {df['text_original'].iloc[i][:180]}...")
print(f"CLEANED:  {df['text_cleaned'].iloc[i][:180]}...")

**Which stopwords did the heavy lifting?** Knowing *what* got removed is how you spot
trouble — if "not" tops this list, your sentiment signal just walked out the door. The
cell below tallies every stopword that was actually removed (on a 2,000-review sample
for speed) and prints the top 20.


In [ ]:
removed_counter = Counter()
for text in df["text_original"].sample(2000, random_state=42):
    removed_counter.update(w for w in text.lower().split() if w in STANDARD_STOPWORDS)

print("📊 MOST FREQUENTLY REMOVED STOPWORDS (2,000-review sample)")
for word, count in removed_counter.most_common(20):
    print(f"   {word:12} {count:6,}  " + "█" * (count // 500))

### 🧪 Experiment

- Feed `remove_stopwords()` a sentence YOU write — try `remove_stopwords("I did not like it, but I do not hate it either", STANDARD_STOPWORDS)`. How much of your meaning survived?
- Change `i = 0` in the before/after cell to a few other review indices. Find one review that cleaning made *misleading* — save it, it's Q1 gold.
- Bump the sample from 2,000 to 10,000 reviews in the tally cell. Does the top-20 list change at all? What does that tell you about how stable stopword statistics are?


**Pause and look (feeds Q1):** scan that top-20 list. Which of those "meaningless" words
might actually carry meaning in restaurant reviews? (`not`? `no`? `very`? `off`?)


## Part 4: Domain Stopwords (10 pts)

**This is where analysts add value.** The standard list is generic English. YOUR corpus
has words that are everywhere but say nothing — in Yelp reviews, "food" and "place"
appear in most reviews yet distinguish nothing. Every domain has these: "patient" in
clinical notes, "account" in banking tickets, "policy" in insurance claims.

First, surface the candidates — high-frequency survivors of standard cleaning:


In [ ]:
all_words = " ".join(df["text_cleaned"].sample(5000, random_state=42)).split()
word_freq = Counter(all_words)

print("📊 TOP 40 WORDS AFTER STANDARD CLEANING — domain-stopword candidates")
for i, (word, count) in enumerate(word_freq.most_common(40), 1):
    print(f"{i:2}. {word:15} {count:6,}")

Now the judgment call. A good domain stopword (1) appears in a large share of
documents, (2) doesn't help tell documents apart, (3) is domain furniture, not signal.
**"good" and "bad" are NOT stopwords here** — they're the sentiment you're trying to
keep. The list below is a defensible set for Yelp restaurant reviews, with the reasoning
inline — read every comment, because in the Playground you'll build this list for YOUR
domain and defend each choice (that's Q2).


In [ ]:
# A defensible domain-stopword list for Yelp restaurant reviews — each with a WHY
DOMAIN_STOPWORDS = {
    "food",       # nearly every review mentions it; distinguishes nothing
    "place",      # generic venue word
    "restaurant", # we know it's a restaurant
    "get", "got", # generic verbs, no signal
    "go", "went", "going",  # everyone "went" somewhere
    "one",        # usually a filler quantity
    "would",      # modal filler ("would recommend" loses little without it)
    "really",     # intensity filler
    "like",       # mostly filler in casual review prose
    "time",       # "first time", "next time" — furniture
    "us",         # pronoun leftover
}

COMBINED_STOPWORDS = STANDARD_STOPWORDS | DOMAIN_STOPWORDS
df["text_domain_cleaned"] = df["text_original"].apply(
    lambda x: remove_stopwords(x, COMBINED_STOPWORDS))
df["word_count_domain"] = df["text_domain_cleaned"].str.split().str.len()

print(f"Standard stopwords: {len(STANDARD_STOPWORDS)} | domain adds: {len(DOMAIN_STOPWORDS)}")
print("\n📊 ORIGINAL vs STANDARD vs DOMAIN")
for name, col in [("Original", "word_count_original"),
                  ("Standard cleaned", "word_count_cleaned"),
                  ("Domain cleaned", "word_count_domain")]:
    print(f"   {name:18} {df[col].sum():>12,} words  (avg {df[col].mean():.1f}/doc)")

### 🧪 Experiment

- Add `"delicious"` to `DOMAIN_STOPWORDS`, re-run this cell and the Part 6 word cloud — what disappears from panel 3, and was that furniture or signal? (Careful: is "delicious" more like "food" or more like "good"?)
- Delete `"really"` and `"like"` from the set and re-run. Does the ORIGINAL/STANDARD/DOMAIN word-count table barely move? What does that say about how much each individual stopword matters?
- Pick TWO more candidates from the top-40 list above, add them with a one-line WHY comment each, and re-run. Could you defend both choices to a skeptical manager?


## Part 5: Negation Problem (10 pts)

**Critical insight: sometimes cleaning destroys the signal.** The standard NLTK stopword
list includes *not, no, nor, don't, won't…* — so "not good" becomes "good" and a 1-star
rant reads like a 5-star rave. For any sentiment task this is a silent catastrophe: the
pipeline runs fine, the numbers come out, and they're wrong.

Watch it happen:


In [ ]:
negation_words = {"not", "no", "never", "neither", "nobody", "none", "nor", "n't",
                  "don't", "doesn't", "didn't", "won't", "wouldn't", "can't", "couldn't"}
print("⚠️ Negations hiding in the standard stopword list:",
      sorted(negation_words & STANDARD_STOPWORDS))

for text in ["This food is not good at all",
             "I would never come back here",
             "No complaints — nothing was wrong"]:
    print(f"\nOriginal: {text}")
    print(f"Cleaned:  {remove_stopwords(text, STANDARD_STOPWORDS)}   ⚠️ meaning flipped?")

How big is this problem in the data? The next cell **measures** it (share of reviews
containing a negation, and whether negation-heavy reviews really do rate lower), then
builds the fix: a **smart** stopword list — combined cleaning power, negations preserved
— and shows one negation-heavy review three ways.


In [ ]:
def contains_negation(text):
    t = str(text).lower()
    return any(neg in t for neg in ["not ", "no ", "never ", "n't"])

df["has_negation"] = df["text_original"].apply(contains_negation)
n, p = df["has_negation"].sum(), df["has_negation"].mean() * 100
print(f"📊 Reviews containing negation: {n:,} ({p:.1f}%)")
print(f"   Avg stars WITH negation:    {df.loc[df['has_negation'], label_column].mean():.2f}")
print(f"   Avg stars WITHOUT negation: {df.loc[~df['has_negation'], label_column].mean():.2f}")

SMART_STOPWORDS = COMBINED_STOPWORDS - negation_words
df["text_smart_cleaned"] = df["text_original"].apply(
    lambda x: remove_stopwords(x, SMART_STOPWORDS))
print(f"\n✅ Smart list: {len(SMART_STOPWORDS)} stopwords (negations preserved)")

example = df.loc[df["has_negation"], "text_original"].iloc[0]
print(f"\nOriginal:       {example[:120]}...")
print(f"Standard clean: {remove_stopwords(example, STANDARD_STOPWORDS)[:120]}...")
print(f"Smart clean:    {remove_stopwords(example, SMART_STOPWORDS)[:120]}...")

### 🧪 Experiment

- Add `"hardly "` and `"barely "` to the detection list in `contains_negation` — they negate without being on any stopword list. How many more reviews get flagged, and does the star-gap widen?
- Reviews that DOUBLE-negate ("not bad at all") break the simple story. Run `df.loc[df['text_original'].str.lower().str.contains('not bad'), label_column].mean()` — do "not bad" reviews rate high or low?
- Preserve only `{"not", "never"}` instead of the full `negation_words` set when building SMART_STOPWORDS. Rerun the three-way example — is the shorter list good enough?


**Pause and look (feeds Q3):** negation-heavy reviews should average noticeably fewer
stars. That gap is exactly the signal standard cleaning throws away.


## Part 6: Word Cloud (6 pts)

One picture to sell your pipeline to a non-technical stakeholder: the same corpus at three
cleaning levels. Original should be drowning in "the/and/was"; standard cleaning surfaces
domain furniture; domain+smart cleaning should surface actual opinions.


In [ ]:
from wordcloud import WordCloud

versions = [("Original", "text_original"),
            ("Standard stopwords removed", "text_cleaned"),
            ("Domain + smart cleaning", "text_smart_cleaned")]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, col) in zip(axes, versions):
    text = " ".join(df[col].sample(1000, random_state=42))
    wc = WordCloud(width=400, height=300, background_color="white", max_words=50).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(title, fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 🧪 Experiment

- Change `max_words=50` to `max_words=120`. Does panel 3 get more informative or just noisier — where's the sweet spot for a slide deck?
- Split panel 3 by sentiment: build one cloud from `df[df[label_column] <= 1]['text_smart_cleaned']` (1-2 star) and one from `df[df[label_column] >= 3]` (4-5 star). Do complaints and raves use different vocabularies?
- Change `random_state=42` in the `.sample()` call and re-run. How much does the cloud change on a different 1,000 reviews — would you trust a word cloud built from a small sample?


**In a markdown cell:** what changed between panel 2 and panel 3, and which panel would
you show a Yelp restaurant owner who asked "what are customers actually saying?"


## 🎮 Your Domain Playground

Now the real thing. **Pick a domain you're passionate about** — and find TEXT from it:
reviews of products YOU actually use (sneakers, games, headphones), posts from a
subreddit you read daily, song lyrics from your favorite genre, match threads from your
team's forum, tickets from your own job. You know that world's vocabulary — that makes
YOU the referee of which words are furniture and which are signal.

1. **Load your text.** Pick ONE loader below and adapt:


In [ ]:
# Pick ONE loader and adapt:

# (a) Hugging Face dataset — search huggingface.co/datasets for your domain
#     (e.g. app reviews, Amazon product categories, song lyrics, game reviews)
# my_df = load_dataset("SOME/dataset-id", split="train[:20000]").to_pandas()
# my_text_column, my_label_column = "text", "label"   # set to the real column names!

# (b) Your own CSV (subreddit export, scraped reviews, your company's tickets)
# from google.colab import files
# import io
# up = files.upload()
# my_df = pd.read_csv(io.BytesIO(up[list(up.keys())[0]]))
# my_text_column, my_label_column = "YOUR_TEXT_COL", "YOUR_RATING_COL"

# (c) Kaggle (needs kaggle.json API token)
# !pip install -q kaggle && mkdir -p ~/.kaggle  # upload kaggle.json, chmod 600
# !kaggle datasets download -d USER/DATASET -p /content --unzip
# my_df = pd.read_csv("/content/FILE.csv")

# print(my_df.columns) FIRST — then rerun Parts 3-6 with my_df

2. **Re-run Parts 3–6 on your corpus** (copy the cells down here; `df` → `my_df`,
   `text_column` → your text column). Build a NEW `DOMAIN_STOPWORDS` for this domain —
   at least 10 words, each with a one-line WHY comment. The Yelp list will NOT fit:
   "food" means nothing in sneaker reviews, but "pair" and "size" might be your furniture.
3. **Check the negation problem in YOUR domain** (Part 5's cells): what share of your
   documents negate, and does your domain even have a rating column to test the gap?
4. **The insider test (markdown cell, 4–6 sentences):** compare your word clouds to the
   demo's. As a domain insider, does panel 3 look like what people in your world actually
   talk about? Which domain stopwords were obvious to you but would be invisible to an
   outsider (that's the point!)? And did any slang or jargon survive cleaning that a
   generic pipeline would have mangled?


## 🤔 Debrief

1. Your cleaning pipeline is deployed and a year of new reviews flows through it. What
   changes in the language (new slang, new menu items, a rebrand) would silently degrade
   it — and who should own updating the stopword lists?
2. Modern transformer models (BERT, GPT) are usually fed **raw, uncleaned** text. Why
   does stopword removal help a word-count analysis but hurt a transformer? When is this
   lab's technique still the right tool?


## 📤 Submit (HW3 — 40 points)

**Deliverables (submit through Canvas):**
- Completed `.ipynb` executed top-to-bottom, INCLUDING the Playground run on your own domain's text (word clouds visible for both corpora)

**Answer these questions in a markdown cell at the end of your notebook:**
- **Q1:** Which removed stopwords might carry meaning in your domain?
- **Q2:** Why did you choose each domain-specific stopword?
- **Q3:** When should you preserve vs. remove negations?
- **Q4:** Example where aggressive text cleaning would HURT analysis

**Grading:**

| Part | Points |
|---|---|
| Part 1-2: Setup and Data | 6 |
| Part 3: Standard Stopwords | 8 |
| Part 4: Domain Stopwords | 10 |
| Part 5: Negation Problem | 10 |
| Part 6: Word Cloud | 6 |

**Due Sunday 10:00 p.m.** — see the Canvas assignment for the exact date. Late work is not accepted (see syllabus).

*AI policy: AI tools are allowed and encouraged. Add one line acknowledging what you used, and be ready to explain every part of what you submit.*
